# 법률 QA 데이터 전처리 — AIHub → 알파카 포맷 변환

**목표**: `data/raw/` 하위 JSON 파일을 재귀 탐색하여 알파카 포맷으로 변환 후 HuggingFace에 업로드

| 지원 포맷 | 키 구조 | 해당 법령 |
|---|---|---|
| label | `data['label']['input/output']` | 형사법, 행정법 |
| taskinfo | `data['taskinfo']['input/output']` | 민사법, 지식재산권법 |

**폴더 구조**: `data/raw/형사법_라벨링데이터/하위폴더/*.json` → 첫 번째 단어(`형사법`)를 `input` 값으로 사용

In [1]:
import os
import json
import pandas as pd
from pathlib import Path
from datasets import Dataset
from dotenv import load_dotenv

c:\Users\SMT21\Desktop\gitMaster\fintech-dev-portfolio\01. AI\vibe_claude\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# .env 로드 (notebooks/ 기준 상위 → vibe_claude/.env)
load_dotenv(Path.cwd().parent / ".env")

HF_TOKEN        = os.getenv("HF_TOKEN")
HF_DATASET_REPO = os.getenv("HF_DATASET_REPO", "yunhwa/legal_qa")

ROOT_DIR      = Path.cwd().parent           # vibe_claude/
RAW_DIR       = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
OUTPUT_FILE   = PROCESSED_DIR / "alpaca_legal.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"RAW_DIR    : {RAW_DIR}")
print(f"OUTPUT_FILE: {OUTPUT_FILE}")
print(f"HF_REPO    : {HF_DATASET_REPO}")
print(f"HF_TOKEN   : {'설정됨' if HF_TOKEN else '미설정 — .env 확인 필요'}")

RAW_DIR    : c:\Users\SMT21\Desktop\gitMaster\fintech-dev-portfolio\01. AI\vibe_claude\data\raw
OUTPUT_FILE: c:\Users\SMT21\Desktop\gitMaster\fintech-dev-portfolio\01. AI\vibe_claude\data\processed\alpaca_legal.json
HF_REPO    : yunhwa/law_instruct
HF_TOKEN   : 설정됨


In [3]:
all_rows   = []
skipped    = 0
file_count = 0

for top_folder in sorted(RAW_DIR.iterdir()):
    if not top_folder.is_dir():
        continue

    # 폴더명 첫 번째 단어 → input 값  (형사법_라벨링데이터 → 형사법)
    domain = top_folder.name.split("_")[0]

    json_files = list(top_folder.rglob("*.json"))
    print(f"[{domain}] {top_folder.name} — {len(json_files)}개 파일")

    for json_file in json_files:
        file_count += 1
        try:
            try:
                with open(json_file, "r", encoding="utf-8-sig") as f:
                    raw = json.load(f)
            except UnicodeDecodeError:
                with open(json_file, "r", encoding="ms949") as f:
                    raw = json.load(f)
        except Exception as e:
            print(f"  [SKIP] {json_file.name}: {e}")
            skipped += 1
            continue

        items = raw if isinstance(raw, list) else [raw]

        for item in items:
            try:
                if "label" in item:
                    instruction = item["label"]["input"]
                    output      = item["label"]["output"]
                elif "taskinfo" in item:
                    instruction = item["taskinfo"]["input"]
                    output      = item["taskinfo"]["output"]
                else:
                    skipped += 1
                    continue

                all_rows.append({
                    "instruction": instruction,
                    "input":       domain,
                    "output":      output,
                })
            except (KeyError, TypeError):
                skipped += 1

print(f"\n파일 {file_count}개 처리 → 레코드 {len(all_rows):,}개 / 스킵 {skipped}개")

[민사법] 민사법_라벨링데이터 — 75624개 파일
[지식재산권법] 지식재산권법_라벨링데이터 — 76160개 파일
[행정법] 행정법_라벨링데이터 — 53989개 파일
[형사법] 형사법_라벨링데이터 — 47434개 파일

파일 253207개 처리 → 레코드 253,207개 / 스킵 0개


In [4]:
# 알파카 JSON 저장
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {OUTPUT_FILE}  ({len(all_rows):,}개 레코드)")

# 미리보기
df = pd.DataFrame(all_rows)
print("\n도메인별 분포:")
print(df["input"].value_counts().to_string())
df.head(3)

저장 완료: c:\Users\SMT21\Desktop\gitMaster\fintech-dev-portfolio\01. AI\vibe_claude\data\processed\alpaca_legal.json  (253,207개 레코드)

도메인별 분포:
input
지식재산권법    76160
민사법       75624
행정법       53989
형사법       47434


,instruction,input,output
0,판결서의 비실명 처리에 대한 규정은 무엇인가요?,민사법,법원사무관은 민사소송법 제163조의2제3항에 따라 판결서에 나타난 개인정보를 보호하...
1,민사집행법에서 집행관의 권한은 무엇인가요?,민사법,"민사집행법 제6조에 따르면, 집행관은 집행을 위해 필요시 채무자의 주거, 창고 등을..."
2,가정법원의 심판이 확정된 경우 그 효력은 언제 발생하나요?,민사법,가정법원의 심판은 판결을 받을 사람이 심판을 고지받음으로써 효력이 발생합니다. 이는...


In [5]:
if not HF_TOKEN:
    raise EnvironmentError(".env에 HF_TOKEN이 설정되지 않았습니다.")

hf_dataset = Dataset.from_pandas(df)

print(f"HuggingFace 업로드 시작 → {HF_DATASET_REPO}")
hf_dataset.push_to_hub(HF_DATASET_REPO, token=HF_TOKEN)
print("업로드 완료!")

HuggingFace 업로드 시작 → yunhwa/law_instruct


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  4.42ba/s]
Processing Files (1 / 1): 100%|██████████| 57.1MB / 57.1MB, 6.21MB/s  
New Data Upload: 100%|██████████| 57.1MB / 57.1MB, 6.21MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:11<00:00, 11.68s/ shards]


업로드 완료!
